In [2]:
!pip install pennylane


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 102.1 MB/s eta 0:00:00


In [37]:
import pennylane as qml
from pennylane import numpy as np

Modelo de Ising básico con PennyLane:implementar el Hamiltoniano de Ising más simple:2 espines (1D), interacción entre vecinos, campo magnético externo: H=−JZ0​Z1​−h(Z0​+Z1​)

Definimos el sistema cuántico (N qubits)

In [38]:
J = 1.0   # interacción
h = 0.5   # campo externo
N = 12
dev = qml.device("default.qubit", wires=N) #no hardware real, 2 qubits

Definimos el Hamiltoniano de Ising

In [39]:
def ising_hamiltonian(N, J, h):

    coeffs = []
    obs = []

    # interacción Z_i Z_{i+1}
    for i in range(N - 1):
        coeffs.append(-J)
        obs.append(qml.PauliZ(i) @ qml.PauliZ(i + 1))

    # campo externo Z_i
    for i in range(N):
        coeffs.append(-h)
        obs.append(qml.PauliZ(i))

    return qml.Hamiltonian(coeffs, obs)

Circuito cuántico simple (estado base variacional)

In [40]:
@qml.qnode(dev)
def circuit(params):

    # inicialización en superposición
    for i in range(N):
        qml.Hadamard(wires=i)

    # capas variacionales
    for i in range(N):
        qml.RY(params[i], wires=i)

    # entrelazamiento en cadena
    for i in range(N - 1):
        qml.CNOT(wires=[i, i + 1])

    H = ising_hamiltonian(N, J, h)

    return qml.expval(H)

Función de coste (energía del sistema)

In [41]:
def cost(params):
    return circuit(params)

Optimización (encontrar estado de mínima energía)

In [42]:
params = np.random.randn(N, requires_grad=True)

opt = qml.GradientDescentOptimizer(stepsize=0.1)

for i in range(100):

    params = opt.step(cost, params)

    if i % 10 == 0:
        print(f"Iter {i} | Energía = {cost(params):.6f}")

Iter 0 | Energía = 2.892125
Iter 10 | Energía = -2.762383
Iter 20 | Energía = -8.286721
Iter 30 | Energía = -11.096010
Iter 40 | Energía = -11.948919
Iter 50 | Energía = -11.997984
Iter 60 | Energía = -11.999922
Iter 70 | Energía = -11.999997
Iter 80 | Energía = -12.000000
Iter 90 | Energía = -12.000000


SOLUCIÓN EXACTA


In [43]:
import numpy as np
import itertools
N = 12
J = 1.0
h = 0.5
def energy(spins, J, h):
    E = 0.0

    # interacción vecinos
    for i in range(N - 1):
        E += -J * spins[i] * spins[i + 1]

    # campo externo
    for i in range(N):
        E += -h * spins[i]

    return E
configs = list(itertools.product([-1, 1], repeat=N))
energies = []

for c in configs:
    e = energy(c, J, h)
    energies.append((e, c))
min_energy, best_config = min(energies, key=lambda x: x[0])

print("Energía mínima exacta:", min_energy)
print("Configuración óptima:", best_config)

Energía mínima exacta: -17.0
Configuración óptima: (1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1)


1. Solución exacta clásica
exploras todo el espacio 2^N, encuentras mínimo global exacto, es exponencial pero exacto

2. Simulación cuántica
no exploras todo el espacio, usas optimización variacional, es aproximado

El ansatz variacional no alcanza el estado fundamental exacto del sistema, lo que evidencia limitaciones en la capacidad expresiva del circuito cuántico y la necesidad de estructuras más profundas o algoritmos específicos como QAOA

PROBAR OTRO OPTIMIZADOR